# Long-term simulation (winter wheat)

`Lintul5Model` simulates **one season per call**. This tutorial runs
**24 consecutive winter-wheat seasons (2000–2024)** over the Brandenburg
long-term weather record, with a sowing date that changes every year and a
nitrogen schedule that follows it.

`torchcrop.longterm` provides the season scheduling around the model. The
day step itself is the ordinary torchcrop one, so a one-season run through
`LongTermSimulator` reproduces `Lintul5Model.forward` bit-for-bit.

| Piece                | Role                                                            |
| -------------------- | --------------------------------------------------------------- |
| `CropCalendar`       | when each crop is sown, and what ends its season                |
| `ManagementSchedule` | irrigation and fertiliser, anchored to each season's sowing day |
| `LongTermSimulator`  | runs the seasons; returns per-season records                    |

**The steps below:** load the weather → build the calendar → add management →
run → inspect the results.


## 1. Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from torchcrop import (
    CropParameters,
    Lintul5Model,
    SiteParameters,
    SoilParameters,
    WeatherDriver,
)
from torchcrop.longterm import (
    CropCalendar,
    LongTermSimulator,
    ManagementEvent,
    ManagementSchedule,
)

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = "Times New Roman"

DATA_DIR = Path("../data", "brandenburg", "torchcrop")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load the 25-year weather record

`weather_longterm/` holds one CSV per location, each a continuous daily series
from 2000 to 2024.

A long-term run needs **one continuous series covering every season**, not one
file per season. It also needs the calendar date of simulated day `0`, so that
real sowing dates can be turned into day indices.


In [ ]:
SOIL_MAP = {
    "SMDRY": "wcad", "SMW": "wcwp", "SMFC": "wcfc", "SMO": "wcst",
    "CRAIRC": "crairc", "SMI": "wci", "SMLOWI": "wci_lower",
    "RDMSO": "rdmso", "RUNFR": "runfr", "CFEV": "cfev", "KSUB": "ksub",
    "NMINS": "nmini", "PMINS": "pmini", "KMINS": "kmini",
    "RTNMINS": "rtnmins", "RTPMINS": "rtpmins", "RTKMINS": "rtkmins",
}
SITE_MAP = {
    "LATITUDE": "latitude", "ALTITUDE": "altitude",
    "IDPL": "idpl", "CO2": "co2",
}
WEATHER_COLS = [
    "Date",           # -> doy
    "TempMean",       # -> davtmp (recomputed as (TempMin + TempMax)/2 below)
    "TempMin",        # -> tmin
    "TempMax",        # -> tmax
    "Radiation",      # -> irrad
    "Precipitation",  # -> rain
    "VapPressure",    # -> vp
    "Windspeed",      # -> wind
]


def load_weather(path):
    """Read one long-term weather CSV into torchcrop channel order."""
    df = pd.read_csv(path, parse_dates=["Date"])[WEATHER_COLS].copy()
    dates = df["Date"]
    # SIMPLACE drives phenology with TMPA = (TMIN + TMAX)/2, not the
    # measured TempMean; match it to reproduce the reference.
    df["TempMean"] = (df["TempMin"] + df["TempMax"]) / 2.0
    df["Radiation"] = df["Radiation"] / 1000.0   # kJ -> MJ m-2 d-1
    df["Date"] = df["Date"].dt.dayofyear          # date -> day-of-year
    return torch.as_tensor(df.values, dtype=torch.float32), dates


def load_table(path, mapping, locations):
    """Read soil.csv / site.csv into a batched parameter dict."""
    table = pd.read_csv(path).set_index("location")
    return {
        field: torch.tensor(
            [float(table.loc[loc][col]) for loc in locations], dtype=torch.float32
        )
        for col, field in mapping.items()
    }

In [ ]:
soil_table = pd.read_csv(DATA_DIR / "soil" / "soil.csv").set_index("location")
site_table = pd.read_csv(DATA_DIR / "site" / "site.csv").set_index("location")
locations = list(soil_table.index.intersection(site_table.index))

series, date_index = zip(
    *(load_weather(DATA_DIR / "weather_longterm" / f"{loc}.csv") for loc in locations)
)
weather = WeatherDriver(torch.stack(series)).to(device=device)
dates = date_index[0]
START_DATE = dates.iloc[0].date()

soil_params = SoilParameters(
    **load_table(DATA_DIR / "soil" / "soil.csv", SOIL_MAP, locations)
).to(device=device)
site_params = SiteParameters(
    **load_table(DATA_DIR / "site" / "site.csv", SITE_MAP, locations)
).to(device=device)

print(f"{weather.batch_size} locations x {weather.n_days} days")
print(f"{START_DATE} to {dates.iloc[-1].date()}")

## 3. Build the crop calendar

A `CropCalendar` says when each crop is sown and what ends its season. Sowing
days are stored as **day indices into the weather series** (`0` = the first
simulated day); the constructors convert real dates for you.

Winter wheat in Brandenburg is sown from late September to mid-October, and the
exact day shifts with the autumn weather. `CropCalendar.annual` takes a
**different day-of-year per year**, which is the point of this tutorial — the
schedule is not the same every year.

Seasons end at **maturity** (`DVS >= 2`) by default. Whatever the rule, the
simulator always clears the field before the next sowing, so seasons can never
overlap.


In [ ]:
YEARS = list(range(2000, 2024))   # 24 autumn sowings; each harvests the next summer

# A plausible sowing window that moves from year to year (DOY 266-287,
# i.e. ~22 Sept to ~14 Oct). Replace this with your own observed dates.
rng = np.random.default_rng(7)
sow_doys = rng.integers(266, 288, size=len(YEARS)).tolist()

calendar = CropCalendar.annual(
    sow_doy=sow_doys,
    n_years=len(YEARS),
    start_date=START_DATE,
    n_days=weather.n_days,
    batch_size=weather.batch_size,
)

print(f"{calendar.n_seasons} seasons")
for year, doy, day in zip(YEARS[:5], sow_doys[:5], calendar.sow_days[0, :5].tolist()):
    print(f"  {year}: DOY {doy} -> day index {day}")

!!! tip "Using your own sowing dates"
If you have observed dates, pass them straight in — one list per location,
so sowing can differ between sites as well as between years:

    ```python
    calendar = CropCalendar.from_dates(
        [["2000-10-02", "2001-09-28", ...],   # location 0
         ["2000-10-05", "2001-10-01", ...]],  # location 1
        start_date=START_DATE,
        n_days=weather.n_days,
    )
    ```


## 4. Add fertiliser and irrigation

Management events are anchored with `days_after_sowing`, so they **re-anchor to
each season's own sowing day** and follow a date that moves. (The model's
built-in `ferntab`/`irrtab` tables are keyed by day-of-year, so they would
repeat identically every year and could not do this.)

Amounts are **g m⁻² of elemental N, P or K** — divide a kg ha⁻¹ recommendation
by 10, so `60 kg N ha⁻¹` is `6.0`.


In [ ]:
schedule = ManagementSchedule(
    fertilizer=[
        ManagementEvent(amount=4.0, days_after_sowing=0),     # 40 kg N/ha at sowing
        ManagementEvent(amount=6.0, days_after_sowing=190),   # 60 kg N/ha at spring tillering
        ManagementEvent(amount=4.0, days_after_sowing=225),   # 40 kg N/ha at booting
    ],
    # Brandenburg wheat is mostly rain-fed; a supplementary application at
    # stem elongation is included here to show how irrigation is specified.
    irrigation=[
        ManagementEvent(amount=20.0, days_after_sowing=210),  # 20 mm
    ],
)

irrigation, fertilizer = schedule.expand(calendar, device=device)
print(f"irrigation array {tuple(irrigation.shape)}, fertilizer array {tuple(fertilizer.shape)}")
print(f"N applied per season: {float(fertilizer[0, :, 0].sum()) / calendar.n_seasons:.0f} g m-2 "
      f"({float(fertilizer[0, :, 0].sum()) / calendar.n_seasons * 10:.0f} kg ha-1)")

## 5. Run the simulation

`iopt = 3` runs water- **and** nitrogen-limited production, so both the
irrigation and the fertiliser matter, and the nitrogen left in the soil at
harvest carries into the following season.

Two arguments keep a long run cheap:

- `collect` names the daily variables worth keeping — everything else is
  dropped. The per-season summaries always come back.
- `torch.no_grad()` — scenario runs need no gradients, and memory then stops
  growing with the length of the run.


In [ ]:
crop_params = CropParameters(crop_name="wheat")
crop_params.iopt = torch.tensor(3.0)      # water- and nitrogen-limited
crop_params = crop_params.to(device=device)

model = Lintul5Model(crop_params, soil_params, site_params).to(device)
simulator = LongTermSimulator(model)

with torch.no_grad():
    output = simulator.run(
        weather,
        calendar,
        management=schedule,
        collect=(
            # crop state — cleared at each harvest
            "dvs", "lai", "tsum", "wso", "rootd",
            # soil state — carried across the harvest boundary
            "smact", "wa", "wa_lower", "nmint", "nmin",
        ),
    )

print(f"simulated {int(output.seasons.valid.sum())} seasons "
      f"({weather.batch_size} locations x {calendar.n_seasons} years)")

## 6. Inspect the results

`output.seasons` is a `SeasonRecord` — one `[locations, seasons]` tensor per
variable. `to_dataframe()` flattens it into a tidy table, one row per
location-season.


In [ ]:
table = output.seasons.to_dataframe(calendar)
table[[
    "batch", "season", "sow_date", "harvest_date", "duration",
    "reached_maturity", "yield_", "max_lai", "tran_cum", "nuptr_cum",
]].head(8)

In [ ]:
seasons = output.seasons
ran = seasons.valid > 0

print(f"seasons reaching maturity : {float(seasons.reached_maturity[ran].mean()) * 100:.0f}%")
print(f"season length [d]         : {float(seasons.duration[ran].min()):.0f} "
      f"- {float(seasons.duration[ran].max()):.0f}")
print(f"yield [g m-2]             : {float(seasons.yield_[ran].mean()):.0f} "
      f"(range {float(seasons.yield_[ran].min()):.0f} - {float(seasons.yield_[ran].max()):.0f})")
print(f"season transpiration [mm] : {float(seasons.tran_cum[ran].mean()):.0f}")
print(f"season N uptake [g N m-2] : {float(seasons.nuptr_cum[ran].mean()):.1f}")

# How tightly does yield follow water, year to year?
yield_by_year = seasons.yield_.mean(dim=0).cpu().numpy()
tran_by_year = seasons.tran_cum.mean(dim=0).cpu().numpy()
print(f"corr(yield, transpiration) : {np.corrcoef(yield_by_year, tran_by_year)[0, 1]:.2f}")

### Yield across 24 seasons

The band shows the spread across the 18 locations; the line is their mean.


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

harvest_years = [y + 1 for y in YEARS]
yields = seasons.yield_.cpu().numpy()          # [locations, seasons]

axes[0].fill_between(
    harvest_years, yields.min(axis=0), yields.max(axis=0),
    color="#1f77b4", alpha=0.2, label="range across locations",
)
axes[0].plot(harvest_years, yields.mean(axis=0), "o-", color="#1f77b4",
             lw=1.5, ms=4, label="mean")
axes[0].set_ylabel("Yield (WSO)\n[g m$^{-2}$]")
axes[0].set_title("Winter-wheat yield, 24 consecutive seasons")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.3)

axes[1].plot(harvest_years, sow_doys, "o-", color="#d62728", lw=1.5, ms=4)
axes[1].set_ylabel("Sowing day\n[DOY of previous autumn]")
axes[1].set_xlabel("Harvest year")
axes[1].grid(alpha=0.3)

fig.tight_layout()
plt.show()

### Inside the seasons

`output.daily` holds the trajectories named in `collect`. State variables have
`[locations, days + 1]` entries — the leading one is the initial condition —
while diagnostics such as `smact` have one value per simulated day; the helper
below trims both to the same window.

The first four seasons are plotted side by side to show what the harvest
boundary does to each kind of variable:

- **Left, crop state.** Cleared at every harvest. `TSUM` returns to zero, LAI
  to bare soil, rooting depth to the seedling value `rdi`, and each sowing
  starts a fresh crop from the seed reserve.
- **Right, soil state.** Carried straight through. Root-zone moisture and
  total profile water run continuously across the boundary, and the soil
  nitrogen pools keep their balance from one season to the next.

Root-zone moisture is worth watching at the harvest lines. Rooting depth drops
from well over a metre back to `rdi` in a single step, so the water below the
new root front is moved into the lower zone — total profile water is conserved
and `smact` continues without a jump.


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 9), sharex=True)

days = 4 * 365
site = 0
t = np.arange(days)
sow_days = calendar.sow_days[site].cpu().numpy()
harvest_days = seasons.harvest_day[site].cpu().numpy()


def daily(name):
    """Day-aligned trajectory for one collected variable.

    State variables carry a leading initial condition (`T + 1` entries),
    diagnostics one value per simulated day (`T`). Both are trimmed to the
    same day window so they can share an axis.
    """
    values = output.daily[name][site]
    offset = 1 if values.shape[0] > weather.n_days else 0
    return values[offset : offset + days].cpu().numpy()


series = {name: daily(name) for name in
          ("dvs", "tsum", "lai", "rootd", "smact", "nmint", "nmin")}
series["water"] = daily("wa") + daily("wa_lower")

CROP, SOIL = "#1f77b4", "#8c564b"
columns = [
    (CROP, "Crop state — cleared at each harvest", [
        ("dvs", "Development stage\n[-]"),
        ("tsum", "Thermal time\n[°C d]"),
        ("lai", "Leaf area index\n[m$^2$ m$^{-2}$]"),
        ("rootd", "Rooting depth\n[m]"),
    ]),
    (SOIL, "Soil state — carried across the boundary", [
        ("smact", "Root-zone moisture\n[m$^3$ m$^{-3}$]"),
        ("water", "Profile water\n[mm]"),
        ("nmint", "Available soil N\n[g N m$^{-2}$]"),
        ("nmin", "Mineralisable soil N\n[g N m$^{-2}$]"),
    ]),
]

for col, (color, heading, panels) in enumerate(columns):
    for row, (key, label) in enumerate(panels):
        ax = axes[row, col]
        ax.plot(t, series[key], color=color, lw=1.2)
        ax.set_ylabel(label, fontsize=9)
        ax.grid(alpha=0.3)
        for d in sow_days[(sow_days >= 0) & (sow_days < days)]:
            ax.axvline(d, color="#2ca02c", ls="--", lw=1)
        for d in harvest_days[(harvest_days >= 0) & (harvest_days < days)]:
            ax.axvline(d, color="#d62728", ls=":", lw=1)
    axes[0, col].set_title(heading, color=color, fontsize=11)
    axes[-1, col].set_xlabel(f"Day since {START_DATE}")

fig.suptitle(
    "First four seasons — sowing (green dashed), harvest (red dotted)",
    fontsize=12,
)
fig.tight_layout()
plt.show()

The two nitrogen pools are where a multi-decade run shows its character, and
both are worth checking before trusting a long simulation:

- **Mineralisable N (`nmin`) drains and does not refill.** Mineralisation
  runs at `rtnmins · nmini` capped by the current pool, and nothing
  replenishes it, so it empties within the first several seasons and
  mineralisation then stops for the rest of the run. Setting
  `CarryOverPolicy(residue_fraction=...)` returns residue N to this pool and
  keeps it turning over.
- **Available N (`nmint`) accumulates.** The mineral balance is
  `fertiliser + mineralisation − uptake`, with no leaching or denitrification
  term, so any application the crop does not take up simply stays in the
  soil. Over 24 seasons at 140 kg N ha⁻¹ against roughly 64 kg N ha⁻¹ of
  uptake, the pool grows steadily.

Neither matters over a single season, which is the setting the underlying
equations were written for. Over decades they make `soil_minerals="carry"`
optimistic about nutrient supply, so match the nitrogen rate to what the crop
actually removes, enable residue return, or use `soil_minerals="reset"` to give
every season the same starting fertility.
